# QuantAlpha AI — Step 10: Model Persistence + Champion/Challenger Retraining

**What this adds (matches Section 6.10 "Continuous Learning Pipeline" from your original doc):**
1. Actually SAVES the trained XGBoost model to disk (`.pkl`, via joblib) — Steps 7/8 trained
   models only in-memory, never persisted them
2. A proper **champion/challenger framework**: when new data arrives, retrain a "challenger"
   model, evaluate it against the current saved "champion" on the SAME held-out test period,
   and only promote the challenger if it's genuinely better — never blindly replace
3. A model registry (a simple CSV log) tracking every training run's metrics over time —
   the lightweight version of what MLflow would do in production

**Important honesty check, upfront:** this is real, legitimate MLOps infrastructure — the kind
of pipeline a production system needs. But it does NOT create predictive edge that isn't in the
data. Steps 7 and 8 already showed AUC ≈ 0.509 (no better than random) using every reasonable
feature combination we could build for short-term (15-day) prediction. Retraining weekly won't
fix that — it will just very reliably keep confirming it, OR catch it if market structure changes
enough that a real signal eventually emerges. Build this because it's correct engineering
practice, not because it will suddenly produce high accuracy.


## 1. Connect to Drive + setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/QuantAlpha')
os.makedirs('models', exist_ok=True)
os.makedirs('models/registry', exist_ok=True)

!pip install xgboost scikit-learn joblib --quiet

import pandas as pd
import numpy as np
import glob
import joblib
import json
from datetime import datetime
import xgboost as xgb
from sklearn.metrics import accuracy_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

print("Ready.")


Mounted at /content/drive
Ready.


## 2. Rebuild the training dataset (same as Step 8 — relative return target, full feature set)
Reusing the exact same feature engineering as Step 8, so results are directly comparable.


In [2]:
fundamental_scores = pd.read_csv('data/fundamental_scores.csv')
fundamental_scores['symbol_short'] = fundamental_scores['symbol'].str.replace('.NS', '', regex=False)
fscore_lookup = dict(zip(fundamental_scores['symbol_short'], fundamental_scores['piotroski_f_score']))
roce_lookup = dict(zip(fundamental_scores['symbol_short'], fundamental_scores['roce']))

event_classification = pd.read_csv('data/event_classification.csv') if os.path.exists('data/event_classification.csv') else pd.DataFrame()
if len(event_classification):
    event_classification['symbol_short'] = event_classification['symbol'].str.replace('.NS', '', regex=False)
    sentiment_lookup = event_classification.groupby('symbol_short').agg(
        has_structural_flag=('verdict', lambda x: any('STRUCTURAL' in v for v in x)),
        negative_news_count=('headline', 'count')
    ).to_dict('index')
else:
    sentiment_lookup = {}

technical_10yr_files = glob.glob('data_10yr/technical/*.parquet')
symbols_10yr = sorted(f.split('/')[-1].replace('.parquet', '') for f in technical_10yr_files)

HOLDING_DAYS = 15

def get_momentum_features(df, i):
    feats = {}
    for window in [5, 20, 50]:
        if i - window >= 0:
            past_price = df.loc[i - window, 'Close']
            curr_price = df.loc[i, 'Close']
            feats[f'momentum_{window}d'] = (curr_price - past_price) / past_price if past_price != 0 else np.nan
        else:
            feats[f'momentum_{window}d'] = np.nan
    return feats

all_rows = []
for sym in symbols_10yr:
    df = pd.read_parquet(f"data_10yr/technical/{sym}.parquet")
    df['Date'] = pd.to_datetime(df['Date']).dt.tz_localize(None)
    df = df.reset_index(drop=True)
    if len(df) < 300:
        continue

    fscore = fscore_lookup.get(sym, np.nan)
    roce = roce_lookup.get(sym, np.nan)
    sentiment_info = sentiment_lookup.get(sym, {})
    has_structural_flag = sentiment_info.get('has_structural_flag', False)
    negative_news_count = sentiment_info.get('negative_news_count', 0)

    for i in range(200, len(df) - HOLDING_DAYS):
        row = {
            'symbol': sym, 'date': df.loc[i, 'Date'],
            'rsi_14': df.loc[i, 'RSI_14'], 'adx_14': df.loc[i, 'ADX_14'], 'atr_14': df.loc[i, 'ATR_14'],
            'close_to_sma50': (df.loc[i, 'Close'] - df.loc[i, 'SMA_50']) / df.loc[i, 'SMA_50'] if pd.notna(df.loc[i, 'SMA_50']) and df.loc[i, 'SMA_50'] != 0 else np.nan,
            'close_to_sma200': (df.loc[i, 'Close'] - df.loc[i, 'SMA_200']) / df.loc[i, 'SMA_200'] if pd.notna(df.loc[i, 'SMA_200']) and df.loc[i, 'SMA_200'] != 0 else np.nan,
            'piotroski_f_score': fscore, 'roce': roce,
            'has_structural_flag': int(has_structural_flag), 'negative_news_count': negative_news_count,
        }
        row.update(get_momentum_features(df, i))
        entry_price = df.loc[i, 'Close']
        exit_price = df.loc[i + HOLDING_DAYS, 'Close']
        row['stock_fwd_return'] = (exit_price - entry_price) / entry_price
        all_rows.append(row)

data = pd.DataFrame(all_rows)
market_avg_by_date = data.groupby('date')['stock_fwd_return'].transform('mean')
data['excess_return'] = data['stock_fwd_return'] - market_avg_by_date
data['target'] = (data['excess_return'] > 0).astype(int)
data = data.dropna(subset=['rsi_14', 'adx_14', 'atr_14', 'close_to_sma50', 'close_to_sma200',
                            'momentum_5d', 'momentum_20d', 'momentum_50d'])
print(f"Dataset ready: {len(data)} rows")


Dataset ready: 397238 rows


## 3. Define the training + evaluation function (reused for every retrain)
This is the core function the weekly retraining loop will call. Always time-split — trains on
everything up to a cutoff date, evaluates ONLY on data after it.


In [3]:
FEATURES = ['rsi_14', 'adx_14', 'atr_14', 'close_to_sma50', 'close_to_sma200',
            'piotroski_f_score', 'roce', 'has_structural_flag', 'negative_news_count',
            'momentum_5d', 'momentum_20d', 'momentum_50d']

def train_and_evaluate(data, train_cutoff_date, test_end_date=None):
    """Trains on data before train_cutoff_date, evaluates on data after it (up to test_end_date).
    This is what gets called every time we retrain -- past or future runs."""
    train = data[data['date'] < train_cutoff_date].copy()
    if test_end_date:
        test = data[(data['date'] >= train_cutoff_date) & (data['date'] < test_end_date)].copy()
    else:
        test = data[data['date'] >= train_cutoff_date].copy()

    if len(train) < 1000 or len(test) < 100:
        return None, None

    for col in ['piotroski_f_score', 'roce']:
        median_val = train[col].median()
        train[col] = train[col].fillna(median_val)
        test[col] = test[col].fillna(median_val)

    model = xgb.XGBClassifier(
        max_depth=4, n_estimators=300, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, eval_metric='logloss', random_state=42
    )
    model.fit(train[FEATURES], train['target'])

    preds = model.predict(test[FEATURES])
    probs = model.predict_proba(test[FEATURES])[:, 1]
    accuracy = accuracy_score(test['target'], preds)
    auc = roc_auc_score(test['target'], probs)
    majority_baseline = max(test['target'].mean(), 1 - test['target'].mean())

    metrics = {
        'train_cutoff': str(train_cutoff_date),
        'train_rows': len(train),
        'test_rows': len(test),
        'accuracy': round(accuracy, 4),
        'auc': round(auc, 4),
        'majority_baseline': round(majority_baseline, 4),
        'edge_over_baseline': round(accuracy - majority_baseline, 4),
        'trained_at': datetime.now().isoformat()
    }
    return model, metrics

print("Training function defined.")


Training function defined.


## 4. Train the initial CHAMPION model + save it properly
This is the model persistence step that was missing from Steps 7/8 — actually writing to disk.


In [4]:
champion_model, champion_metrics = train_and_evaluate(data, train_cutoff_date='2024-01-01')

if champion_model is not None:
    joblib.dump(champion_model, 'models/champion_model.pkl')
    with open('models/champion_metrics.json', 'w') as f:
        json.dump(champion_metrics, f, indent=2)
    print("Champion model saved to models/champion_model.pkl")
    print(json.dumps(champion_metrics, indent=2))
else:
    print("Not enough data to train.")


Champion model saved to models/champion_model.pkl
{
  "train_cutoff": "2024-01-01",
  "train_rows": 283160,
  "test_rows": 114078,
  "accuracy": 0.532,
  "auc": 0.5091,
  "majority_baseline": 0.5362,
  "edge_over_baseline": -0.0043,
  "trained_at": "2026-07-10T05:14:56.704900"
}


## 5. Champion/Challenger comparison function
This is what a "weekly retrain" would call. It trains a NEW challenger on more recent data,
evaluates both champion and challenger on the SAME test window, and only promotes the challenger
if it's genuinely better -- never blindly swaps in whatever was trained most recently.


In [5]:
def run_challenger_check(data, new_train_cutoff, log_path='models/registry/training_log.csv'):
    """Simulates what a weekly automated retrain would do:
    1. Train a challenger on data up to new_train_cutoff
    2. Load the current champion
    3. Evaluate BOTH on the same recent test window
    4. Promote challenger only if it's meaningfully better (AUC improvement > 0.01,
       a small buffer so we don't swap models over noise)
    5. Log the outcome either way
    """
    challenger_model, challenger_metrics = train_and_evaluate(data, train_cutoff_date=new_train_cutoff)
    if challenger_model is None:
        print("Not enough new data yet for a challenger run.")
        return

    champion_model = joblib.load('models/champion_model.pkl')
    with open('models/champion_metrics.json') as f:
        current_champion_metrics = json.load(f)

    # Evaluate CHAMPION on the SAME test window as the challenger, for a fair comparison
    test = data[data['date'] >= new_train_cutoff].copy()
    for col in ['piotroski_f_score', 'roce']:
        test[col] = test[col].fillna(test[col].median())
    champ_preds = champion_model.predict(test[FEATURES])
    champ_probs = champion_model.predict_proba(test[FEATURES])[:, 1]
    champ_auc_on_new_window = roc_auc_score(test['target'], champ_probs)

    auc_improvement = challenger_metrics['auc'] - champ_auc_on_new_window
    PROMOTION_THRESHOLD = 0.01  # require a small, meaningful margin, not noise

    decision = "PROMOTED" if auc_improvement > PROMOTION_THRESHOLD else "KEPT CHAMPION"

    log_entry = {
        'run_date': datetime.now().isoformat(),
        'train_cutoff': new_train_cutoff,
        'champion_auc_on_new_window': round(champ_auc_on_new_window, 4),
        'challenger_auc': challenger_metrics['auc'],
        'auc_improvement': round(auc_improvement, 4),
        'decision': decision
    }

    log_df = pd.DataFrame([log_entry])
    if os.path.exists(log_path):
        log_df.to_csv(log_path, mode='a', header=False, index=False)
    else:
        log_df.to_csv(log_path, index=False)

    if decision == "PROMOTED":
        joblib.dump(challenger_model, 'models/champion_model.pkl')
        with open('models/champion_metrics.json', 'w') as f:
            json.dump(challenger_metrics, f, indent=2)
        print(f"Challenger PROMOTED. New champion AUC: {challenger_metrics['auc']}")
    else:
        print(f"Challenger did not beat champion by enough margin. Champion kept.")
        print(f"  Champion AUC (on new window): {champ_auc_on_new_window}")
        print(f"  Challenger AUC: {challenger_metrics['auc']}")

    return log_entry

print("Champion/challenger function ready.")


Champion/challenger function ready.


## 6. Simulate what weekly retraining would have looked like, historically
Since we can't wait actual weeks right now, this simulates several past retraining checkpoints
using data you already have -- showing what the log would look like over time.


In [6]:
# Simulate quarterly checkpoints through 2024-2026 (using available data)
simulation_dates = ['2024-04-01', '2024-07-01', '2024-10-01', '2025-01-01', '2025-04-01']

for cutoff in simulation_dates:
    if pd.Timestamp(cutoff) < data['date'].max():
        print(f"\n--- Checkpoint: {cutoff} ---")
        run_challenger_check(data, new_train_cutoff=cutoff)

print("\n\n=== Full training log ===")
log_df = pd.read_csv('models/registry/training_log.csv')
print(log_df.to_string(index=False))



--- Checkpoint: 2024-04-01 ---
Challenger did not beat champion by enough margin. Champion kept.
  Champion AUC (on new window): 0.5073418434705647
  Challenger AUC: 0.5095

--- Checkpoint: 2024-07-01 ---
Challenger did not beat champion by enough margin. Champion kept.
  Champion AUC (on new window): 0.5073006156170152
  Challenger AUC: 0.5072

--- Checkpoint: 2024-10-01 ---
Challenger did not beat champion by enough margin. Champion kept.
  Champion AUC (on new window): 0.5093408277726356
  Challenger AUC: 0.5142

--- Checkpoint: 2025-01-01 ---
Challenger did not beat champion by enough margin. Champion kept.
  Champion AUC (on new window): 0.5066661529458261
  Challenger AUC: 0.5093

--- Checkpoint: 2025-04-01 ---
Challenger did not beat champion by enough margin. Champion kept.
  Champion AUC (on new window): 0.508153652111754
  Challenger AUC: 0.5092


=== Full training log ===
                  run_date train_cutoff  champion_auc_on_new_window  challenger_auc  auc_improvement   

## 7. How to read this honestly

- **Look at the `decision` column across checkpoints.** If it's mostly "KEPT CHAMPION" with tiny
  AUC differences hovering near 0.50-0.52 throughout, that's consistent with everything found in
  Steps 7-8 — no real underlying signal for the model to improve on, regardless of retraining
  frequency.
- **This framework is still valuable** because: (a) it's correct engineering practice matching
  your original doc's vision, (b) IF the market structure changes and real signal emerges later,
  this pipeline would actually catch and adopt it automatically, (c) it prevents a common mistake
  — blindly replacing a working model with a newer one that's actually worse due to noise.
- **To make this truly "weekly" and automatic** (not just simulated here), you'd deploy this
  logic on a real scheduler (cron job, GitHub Actions, or Airflow) running outside Colab — Colab
  itself isn't meant to stay running unattended for scheduled jobs.


## 8. Summary
Saved:
- `models/champion_model.pkl` — the current best model, properly persisted (joblib)
- `models/champion_metrics.json` — its evaluation metrics
- `models/registry/training_log.csv` — full history of every training run and promotion decision

**This completes the MLOps piece from your original doc's Section 6.10 (Continuous Learning
Pipeline)** — properly built, even though the underlying short-term prediction task itself
doesn't show real edge. The honest, complete story: rigorous infrastructure + rigorous validation
+ an honest conclusion about what the data can and can't support.

**To actually automate this weekly in production** (future step, separate from Colab): wrap
Sections 4-5 into a script, schedule it with GitHub Actions (free) or a cron job on a small
server, triggered weekly after new price data is available.
